In [1]:
from paths import *
from nano_maker.nanomaker import NanoMaker
from nano_maker.container.configs import (skeleton_weights, naanobot_weights,
                                          skeleton_config, naanobot_config, radial_config)

A -- [0.44999999999999996, 0, 0.43, 0.41, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1.46, 0.87, 0.71, 1.0]
C -- [0.6, 0, 0.35857142857142854, 0.49, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0.93, 1.37, 0.84, 0.97]
D -- [0.65, -1, 0.21285714285714286, -0.55, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.96, 0.54, 1.31, 0.87]
E -- [0.75, -1, 0.23, -0.31, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1.46, 0.65, 0.85, 0.89]
F -- [0.8500000000000001, 0, 0.39142857142857146, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1.04, 1.44, 0.7, 0.88]
G -- [0.4, 0, 0.42642857142857143, 0.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.42, 0.36, 1.85, 0.96]
H -- [0.8, 0, 0.5421428571428571, 0.08, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.85, 1.16, 1.02, 0.89]
I -- [0.65, 0, 0.42714285714285716, 1.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.86, 1.85, 0.59, 0.76]
K -- [0.75, 1, 0.6764285714285715, -0.23, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1.12, 0.86, 0.99, 1.24]
L -- [0.65, 0, 0.42714285714285716, 0.97, 0, 0, 0, 0, 0, 0, 0, 

In [2]:
skeleton_weight_filename = skeleton_weights
skeleton_cfg = skeleton_config
naanobot_weight_filename = naanobot_weights
naanobot_config = naanobot_config
radial_cfg = radial_config

In [3]:
model = NanoMaker(skeleton_weight_filename=skeleton_weight_filename,
                  naanobot_weight_filename=naanobot_weight_filename,
                  skeleton_cfg=skeleton_config,
                  naanobot_config=naanobot_config,
                  radial_cfg=radial_cfg)

In [4]:
drug_i_want_to_deliver = "CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C"

In [5]:
model.ingest_chemical(drug_i_want_to_deliver)

Chemical Ingested: CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C
Drug Likeness Rules Passed: 4 / 4


In [6]:
pocket_data = model.generate_pocket_data()
print(len(pocket_data))
print(type(pocket_data))

3
<class 'dict'>


In [7]:
pocket_data

{'SMILES': 'CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C',
 '3D_skeleton': [[9.323131550658474,
   -11.967625744693079,
   -0.23373284581855525],
  [10.387204696347945, -9.729483706503729, -2.2229013694413604],
  [9.79254853301095, -6.866697275134815, -7.88182182331398],
  [-1.2759497277590606, -13.300421959672963, -3.588215273179142],
  [13.038001783035064, 0.22514848351495836, -2.8690369752094824],
  [6.174997313748876, -10.972123753597977, -3.300833647978064],
  [10.68016710350466, -2.6732057733003995, 7.000312136070471],
  [2.358478186463093, -12.404058003982112, -1.4718076108601321],
  [4.015172101226681, 10.443470374851865, 5.135953214809498],
  [0.5064086862455738, -12.306298780100672, -1.6864058435015332],
  [0.8731267076750485, 12.180329653569169, -1.135009755860567],
  [-3.822875392528043, -11.45726562451297, 1.2824272931965155],
  [-4.0816580215992095, 8.706241341243615, -7.745570902103627],
  [11.751867618157712, 0.3399056499316234, -3.9877559259405446],
  [-3.5143715

In [8]:
smiles_str = pocket_data["SMILES"]
skeleton = pocket_data['3D_skeleton']
aa_seq = pocket_data['aa_ids']

In [9]:
# GENERATION QUALITY ASSESSMENTS

In [10]:
import torch
from nano_maker.naanobot import NAAnoBot
from nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

@torch.no_grad()
def diagnose_generation(model, map4_enc, sph_coordinates, n=10):
    model.eval()
    map4_enc = map4_enc.to(next(model.parameters()).device)

    bioch_context = [model.naano_module.get_nAAnovector("VOID") for _ in range(model.block_size)]
    coord_context = [[model.max_angstroms, 0, 0] for _ in range(model.block_size)]

    for i, coordinate in enumerate(sph_coordinates[:n]):
        naano_X = model.naano_module.get_nAAno_X(coord_context, bioch_context, coordinate).unsqueeze(0).to(map4_enc.device)
        output, _ = model.forward(naano_X, map4_enc)

        # print raw predicted vector
        print(f"\nPosition {i}:")
        print(f"  raw output: {output[0].detach().cpu().numpy().round(2)}")

        # print distances to all amino acids
        vector = output[0].detach().float()
        distances = {}
        for aa_id, n_v in model.naano_module.nAAno_vectors.items():
            if aa_id == 'VOID':
                continue
            n_v_t = torch.tensor(n_v, dtype=torch.float32)
            distances[aa_id] = torch.norm(vector - n_v_t).item()

        sorted_distances = sorted(distances.items(), key=lambda x: x[1])
        print(f"  top 3: {sorted_distances[:3]}")

        # update context
        aa_id = sorted_distances[0][0]
        bioch_context = bioch_context[1:] + [model.naano_module.get_nAAnovector(aa_id)]
        coord_context = coord_context[1:] + [coordinate.tolist() if torch.is_tensor(coordinate) else coordinate]


map4_enc = torch.tensor(smiles_fingerprint(drug_i_want_to_deliver), dtype=torch.float32).unsqueeze(0)
nb_cfg = naanobot_config.copy()
naanobot_model = NAAnoBot(n_embd=nb_cfg["n_embd"], n_head=nb_cfg["n_head"],
                                           n_layers=nb_cfg["n_layers"],
                                           block_size=nb_cfg["block_size"],
                                           map4_res=nb_cfg["map4_res"],
                                           n_nAAno_features=nb_cfg["n_nAAno_features"],
                                           n_spatial_features=nb_cfg["n_spatial_features"],
                                           max_angstroms=nb_cfg["max_angstroms"],
                                           dropout=nb_cfg["dropout"])
diagnose_generation(model=naanobot_model, map4_enc=map4_enc, sph_coordinates=model._pocket_sph_skeleton())


Position 0:
  raw output: [ 1.68 -0.37  0.16 -0.29  0.01  0.1  -0.29 -1.48 -1.18 -0.82  0.14  0.96
  0.41  0.12 -0.12 -0.7   0.03 -0.3  -0.24 -0.26 -0.23]
  top 3: [('D', 3.7878901958465576), ('E', 3.8198766708374023), ('G', 3.826467275619507)]

Position 1:
  raw output: [ 1.05 -0.   -0.6   0.14 -0.54 -0.71  0.89 -1.12 -0.51  0.09  0.95  0.96
  0.29 -0.01  0.12 -1.55  0.28 -0.22  0.25  0.52  0.32]
  top 3: [('G', 3.406770706176758), ('C', 3.4325833320617676), ('M', 3.442633628845215)]

Position 2:
  raw output: [ 0.79 -0.25 -0.5  -0.22 -0.81 -0.41  0.24 -0.8  -0.82 -0.67  0.16  0.63
  1.07  0.01  0.13 -1.7  -0.05  0.51  0.53  0.08 -0.09]
  top 3: [('F', 3.2861146926879883), ('A', 3.347376823425293), ('W', 3.4028918743133545)]

Position 3:
  raw output: [ 0.41 -0.17  0.03 -0.96 -0.72 -0.37  0.06 -0.27 -0.39 -0.69 -0.37  0.45
  1.03 -0.39  0.13 -1.2  -0.44  0.7  -0.13 -0.71  0.42]
  top 3: [('A', 3.2514240741729736), ('Q', 3.4194979667663574), ('E', 3.482645034790039)]

Position 4:
  ra